# PAD-ONAP — Notebook 1: Multi-class DDoS Detection (CICDDoS2019)

**Thesis:** Proactive AI-Driven DDoS Defense (PAD-ONAP)
**Module:** M2 — AI Detection (baseline comparison)

This notebook replaces the previous `ddos-train-v*` series with a **clean
baseline benchmark** on CICDDoS2019. Forecasting was moved to Notebook 2
(CSE-CIC-IDS2018) because CICDDoS2019 is too benign-poor for time-series
forecasting (>99% attack rows).

## Six baselines

| # | Model        | Library      | Notes                                   |
|---|--------------|--------------|-----------------------------------------|
| 1 | XGBoost      | xgboost      | GPU hist, Optuna-tuned (cached)         |
| 2 | LightGBM     | lightgbm     | GPU, native categorical handling        |
| 3 | RandomForest | scikit-learn | 300 trees, balanced subsample           |
| 4 | ExtraTrees   | scikit-learn | 300 trees, faster, often stronger on DDoS |
| 5 | MLP          | scikit-learn | 2 hidden layers, early stopping         |
| 6 | 1D-CNN       | PyTorch      | Conv1d on 22-d feature vector           |

## Best-paper tricks applied

* **Anti-leak group-split by file** (every CSV stays entirely in train OR test).
* **22 CICFlowMeter window-level features** + **Extra Trees** feature
  importance audit.
* **Robust scaler + `log1p`** on heavy-tail throughput features.
* **SMOTE** only for minority *attack* classes (BENIGN kept real).
* **Per-class threshold calibration** on the validation slice.
* Final metrics reported: **macro-F1, balanced accuracy, per-class F1,
  ROC-AUC (OvR), PR-AUC**.

## Pipeline flow

```
CICDDoS2019 CSVs (Kaggle: rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files)
       │
       ▼  Section 1 — Audit (row counts, mapped labels)
       ▼  Section 2 — Feature extraction (22 features × 100-row windows)
       ▼  Section 3 — Group-split + scale + SMOTE
       ▼  Section 4 — Train 6 baselines, calibrate thresholds
       ▼  Section 5 — Compare, plot, SHAP on best model, save
```


In [7]:
# ── Cell 0: Install / upgrade packages ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('xgboost>=2.0', 'lightgbm>=4.3', 'shap>=0.44',
    'imbalanced-learn>=0.12', 'scikit-learn>=1.4')
print('Packages ready.')


Packages ready.


In [8]:
# ── Cell 1: Imports + GPU check ─────────────────────────────────────────────
import os, glob, time, json, pickle, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import xgboost as xgb
import lightgbm as lgb
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score,
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

DATASET_DIR = Path('/kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files')
if not DATASET_DIR.exists():
    # Fallback: scan for any 2019 dataset under /kaggle/input
    cands = sorted(Path('/kaggle/input').glob('*ddos*2019*')) if Path('/kaggle/input').exists() else []
    if cands:
        DATASET_DIR = cands[0]

OUTPUT_DIR = Path('/kaggle/working/pad_onap_detect')
MODELS_DIR = OUTPUT_DIR / 'models'
FIG_DIR    = OUTPUT_DIR / 'figures'
RESULT_DIR = OUTPUT_DIR / 'results'
for d in (MODELS_DIR, FIG_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dataset dir : {DATASET_DIR}  exists={DATASET_DIR.exists()}')
print(f'Output  dir : {OUTPUT_DIR}')
print(f'PyTorch     : {torch.__version__}  device={DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


Dataset dir : /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files  exists=True
Output  dir : /kaggle/working/pad_onap_detect
PyTorch     : 2.10.0+cu128  device=cuda
GPU         : Tesla T4


---
## Section 1 — Audit

In [ ]:
# ── Cell 2: Label map + feature list ────────────────────────────────────────
# 7-class taxonomy (spec-aligned).  Eight reflection sub-types stay collapsed
# into a single Amplification macro-class to keep the minority share usable
# for SMOTE while preserving the operational distinction (reflection vs
# exploit vs application).

LABEL_MAP = {
    'BENIGN': 0,
    # Reflection / amplification → 1
    'DrDoS_DNS': 1, 'DrDoS_LDAP': 1, 'LDAP': 1,
    'DrDoS_MSSQL': 1, 'MSSQL': 1,
    'DrDoS_NetBIOS': 1, 'NetBIOS': 1,
    'DrDoS_NTP': 1, 'NTP': 1,
    'DrDoS_SNMP': 1, 'SNMP': 1,
    'DrDoS_SSDP': 1, 'SSDP': 1,
    'DrDoS_UDP': 1, 'UDP': 1,
    # Exploitation
    'Syn': 2, 'SYN': 2,
    'UDP-lag': 3, 'UDP-Lag': 3, 'UDPLag': 3,
    # Application layer
    'WebDDoS': 4, 'Web DDoS': 4,
    'TFTP': 5,
    'Portmap': 6,
}
CLASS_NAMES = {
    0: 'BENIGN', 1: 'Amplification', 2: 'Syn', 3: 'UDP-lag',
    4: 'WebDDoS', 5: 'TFTP', 6: 'Portmap',
}
N_CLASSES = len(CLASS_NAMES)

FEATURE_NAMES = [
    'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts',
    'tot_len_fwd', 'tot_len_bwd',
    'fwd_pkt_max', 'fwd_pkt_mean', 'bwd_pkt_mean',
    'flow_bytes_s', 'flow_pkts_s',
    'flow_iat_mean', 'flow_iat_std',
    'fwd_iat_mean', 'bwd_iat_mean',
    'pkt_len_mean', 'pkt_len_std',
    'syn_flag_cnt', 'ack_flag_cnt', 'psh_flag_cnt', 'urg_flag_cnt',
    'down_up_ratio', 'protocol_mode',
]
N_FEATURES = len(FEATURE_NAMES)
assert N_FEATURES == 22

# Heavy-tail features that benefit from log1p
LOG1P_FEATURES = {
    'flow_duration', 'flow_bytes_s', 'flow_pkts_s',
    'flow_iat_mean', 'flow_iat_std', 'fwd_iat_mean', 'bwd_iat_mean',
    'tot_len_fwd', 'tot_len_bwd',
}

print(f'Classes  : {N_CLASSES}')
print(f'Features : {N_FEATURES}')
print(f'log1p    : {sorted(LOG1P_FEATURES)}')


In [10]:
# ── Cell 3: Audit CSV files ─────────────────────────────────────────────────
csv_files = sorted(glob.glob(str(DATASET_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files')
for p in csv_files[:5]:
    print('  ', p)

LABEL_COL_CANDIDATES = [' Label', 'Label', 'label']

def find_label_col(cols):
    for c in LABEL_COL_CANDIDATES:
        if c in cols:
            return c
    return None

all_labels = Counter()
grand_total = 0
file_label_counts = {}

for fp in csv_files:
    try:
        # Stream count only — faster than full load
        cnt_local = Counter()
        for chunk in pd.read_csv(fp, usecols=lambda c: c.strip() == 'Label' or c == ' Label',
                                 chunksize=500_000, low_memory=False, on_bad_lines='skip'):
            col = find_label_col(chunk.columns) or chunk.columns[0]
            cnt_local.update(chunk[col].astype(str).str.strip())
        file_label_counts[fp] = cnt_local
        all_labels.update(cnt_local)
        grand_total += sum(cnt_local.values())
        print(f'  {Path(fp).name:50s}  rows={sum(cnt_local.values()):>10,d}')
    except Exception as exc:
        print(f'  [skip] {Path(fp).name}: {exc}')

print('-' * 80)
class_counts = defaultdict(int)
for lbl, cnt in all_labels.items():
    cls = LABEL_MAP.get(lbl, -1)
    if cls >= 0:
        class_counts[cls] += cnt

print(f'{"Cls":<4}{"Name":<16}{"Rows":>14}{"%":>8}')
for cid in range(N_CLASSES):
    cnt = class_counts.get(cid, 0)
    pct = 100 * cnt / max(grand_total, 1)
    print(f'{cid:<4}{CLASS_NAMES[cid]:<16}{cnt:>14,}{pct:>7.2f}%')
print(f'{"":4}{"TOTAL":<16}{grand_total:>14,}{100:>7.2f}%')


Found 18 CSV files
   /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files/01-12/DrDoS_DNS.csv
   /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files/01-12/DrDoS_LDAP.csv
   /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files/01-12/DrDoS_MSSQL.csv
   /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files/01-12/DrDoS_NTP.csv
   /kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files/01-12/DrDoS_NetBIOS.csv
  DrDoS_DNS.csv                                       rows= 5,074,413
  DrDoS_LDAP.csv                                      rows= 2,181,542
  DrDoS_MSSQL.csv                                     rows= 4,524,498
  DrDoS_NTP.csv                                       rows= 1,217,007
  DrDoS_NetBIOS.csv                                   rows= 4,094,986
  DrDoS_SNMP.csv                                      rows= 5,161,377
  DrDoS_SSDP.csv                  

---
## Section 2 — Feature extraction

In [11]:
# ── Cell 4: Window-level feature extractor ──────────────────────────────────
RAW_TO_FEAT = {
    'flow_duration':   'Flow Duration',
    'tot_fwd_pkts':    'Total Fwd Packets',
    'tot_bwd_pkts':    'Total Backward Packets',
    'tot_len_fwd':     'Total Length of Fwd Packets',
    'tot_len_bwd':     'Total Length of Bwd Packets',
    'fwd_pkt_max':     'Fwd Packet Length Max',
    'fwd_pkt_mean':    'Fwd Packet Length Mean',
    'bwd_pkt_mean':    'Bwd Packet Length Mean',
    'flow_bytes_s':    'Flow Bytes/s',
    'flow_pkts_s':     'Flow Packets/s',
    'flow_iat_mean':   'Flow IAT Mean',
    'flow_iat_std':    'Flow IAT Std',
    'fwd_iat_mean':    'Fwd IAT Mean',
    'bwd_iat_mean':    'Bwd IAT Mean',
    'pkt_len_mean':    'Packet Length Mean',
    'pkt_len_std':     'Packet Length Std',
    'syn_flag_cnt':    'SYN Flag Count',
    'ack_flag_cnt':    'ACK Flag Count',
    'psh_flag_cnt':    'PSH Flag Count',
    'urg_flag_cnt':    'URG Flag Count',
    'down_up_ratio':   'Down/Up Ratio',
    'protocol_mode':   'Protocol',
}

def safe_numeric(series: pd.Series) -> np.ndarray:
    v = pd.to_numeric(series, errors='coerce').to_numpy(dtype=np.float64)
    v = np.where(np.isinf(v) | (v < 0), np.nan, v)
    if np.isnan(v).all():
        return np.zeros_like(v)
    med = np.nanmedian(v)
    return np.where(np.isnan(v), med, v)

def extract_window(df_win: pd.DataFrame) -> np.ndarray:
    out = np.zeros(N_FEATURES, dtype=np.float32)
    for i, feat in enumerate(FEATURE_NAMES):
        raw = RAW_TO_FEAT[feat]
        if raw not in df_win.columns:
            continue
        vals = safe_numeric(df_win[raw])
        if feat == 'protocol_mode':
            v = vals.astype(int)
            out[i] = float(np.bincount(v[v >= 0]).argmax()) if len(v) else 0.0
        else:
            out[i] = float(np.nanmean(vals))
    return out

print('Window extractor ready: %d features per window.' % N_FEATURES)


Window extractor ready: 22 features per window.


In [12]:
# ── Cell 5: Streaming window builder (group key = source file) ──────────────
WINDOW_SIZE        = 100
STEP_ATTACK        = 50
STEP_BENIGN        = 4          # denser sampling for BENIGN
MAX_WINDOWS        = 280_000
MIN_BENIGN_WINDOWS = 60_000

X_list, y_list, grp_list = [], [], []
class_quota = {c: int(MAX_WINDOWS * 0.18) for c in range(N_CLASSES)}
class_quota[0] = MIN_BENIGN_WINDOWS

made = defaultdict(int)
t0 = time.time()

for fp in csv_files:
    if sum(made.values()) >= MAX_WINDOWS:
        break
    try:
        df_iter = pd.read_csv(fp, chunksize=200_000, low_memory=False, on_bad_lines='skip')
    except Exception as exc:
        print(f'[skip] {Path(fp).name}: {exc}'); continue

    fname = Path(fp).name
    buffer = []
    buf_labels = []
    for chunk in df_iter:
        chunk.columns = [c.strip() for c in chunk.columns]
        lbl_col = 'Label' if 'Label' in chunk.columns else chunk.columns[-1]
        chunk['_y'] = chunk[lbl_col].astype(str).str.strip().map(LABEL_MAP)
        chunk = chunk.dropna(subset=['_y'])
        if chunk.empty:
            continue
        buffer.append(chunk)
        if sum(len(b) for b in buffer) < WINDOW_SIZE:
            continue
        df_all = pd.concat(buffer, ignore_index=True)

        i = 0
        while i + WINDOW_SIZE <= len(df_all):
            win = df_all.iloc[i:i+WINDOW_SIZE]
            y_arr = win['_y'].to_numpy(dtype=np.int64)
            # Majority label of the window
            uniq, cnt = np.unique(y_arr, return_counts=True)
            top = int(uniq[cnt.argmax()])
            # Skip if quota hit
            if made[top] >= class_quota[top]:
                step = STEP_BENIGN if top == 0 else STEP_ATTACK
                i += step
                continue
            X_list.append(extract_window(win))
            y_list.append(top)
            grp_list.append(fname)
            made[top] += 1
            step = STEP_BENIGN if top == 0 else STEP_ATTACK
            i += step
            if sum(made.values()) >= MAX_WINDOWS:
                break
        # keep last incomplete window
        buffer = [df_all.iloc[i:]] if i < len(df_all) else []
        if sum(made.values()) >= MAX_WINDOWS:
            break
    print(f'  {fname:48s} cum windows={sum(made.values()):>7,}')

X = np.vstack(X_list).astype(np.float32)
y = np.asarray(y_list, dtype=np.int64)
groups = np.asarray(grp_list)
print(f'\nBuilt {len(X):,} windows in {time.time()-t0:.1f}s')
print('Per class :', {CLASS_NAMES[c]: int(np.sum(y == c)) for c in range(N_CLASSES)})
print('Per file  :', len(set(groups)))


  DrDoS_DNS.csv                                    cum windows= 50,754
  DrDoS_LDAP.csv                                   cum windows= 50,754
  DrDoS_MSSQL.csv                                  cum windows= 50,950
  DrDoS_NTP.csv                                    cum windows= 54,546
  DrDoS_NetBIOS.csv                                cum windows= 54,625
  DrDoS_SNMP.csv                                   cum windows= 54,708
  DrDoS_SSDP.csv                                   cum windows= 54,708
  DrDoS_UDP.csv                                    cum windows= 54,842
  Syn.csv                                          cum windows= 86,494
  TFTP.csv                                         cum windows=142,276
  UDPLag.csv                                       cum windows=150,456
  LDAP.csv                                         cum windows=151,097
  MSSQL.csv                                        cum windows=151,195
  NetBIOS.csv                                      cum windows=151,195
  Port

In [21]:
# ── Cell 6: Per-file temporal split (80/20) with purge gap ─────────────────
# Each CSV in CICDDoS2019 contains ONE attack type, so a pure group-split
# pushes whole classes to one side.  Instead split inside each file (v4 FIX-1c).
TEMPORAL_RATIO = 0.8
PURGE_WINDOWS  = 20   # drop windows straddling the train/test boundary

train_mask = np.zeros(len(X), dtype=bool)
test_mask  = np.zeros(len(X), dtype=bool)

for fname in np.unique(groups):
    idx = np.where(groups == fname)[0]
    n = len(idx)
    if n < 5:
        continue
    cut = int(n * TEMPORAL_RATIO)
    lo = max(cut - PURGE_WINDOWS // 2, 0)
    hi = min(cut + PURGE_WINDOWS // 2, n)
    train_mask[idx[:lo]] = True
    test_mask[idx[hi:]]  = True   # purge gap [lo, hi) is dropped

X_tr, X_te = X[train_mask], X[test_mask]
y_tr, y_te = y[train_mask], y[test_mask]
g_tr, g_te = groups[train_mask], groups[test_mask]
purged = len(X) - train_mask.sum() - test_mask.sum()

print(f'Train : {len(X_tr):,}   files={len(set(g_tr))}')
print(f'Test  : {len(X_te):,}   files={len(set(g_te))}')
print(f'Purged: {purged:,}  (around train/test boundary inside each file)')
print('Per class TRAIN:', {CLASS_NAMES[c]: int(np.sum(y_tr == c)) for c in range(N_CLASSES)})
print('Per class TEST :', {CLASS_NAMES[c]: int(np.sum(y_te == c)) for c in range(N_CLASSES)})

# log1p on heavy-tail columns
log_idx = [i for i, f in enumerate(FEATURE_NAMES) if f in LOG1P_FEATURES]
def apply_log1p(M):
    M = M.copy()
    M[:, log_idx] = np.log1p(np.clip(M[:, log_idx], 0, None))
    return M
X_tr_l = apply_log1p(X_tr)
X_te_l = apply_log1p(X_te)

scaler = RobustScaler(quantile_range=(5, 95))
X_tr_s = scaler.fit_transform(X_tr_l).astype(np.float32)
X_te_s = scaler.transform(X_te_l).astype(np.float32)
print('Scaled.  Train mean ~0, std ~1 :', X_tr_s.mean(), X_tr_s.std())

with open(MODELS_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'log_idx': log_idx,
                 'features': FEATURE_NAMES,
                 'split_method': 'per_file_temporal_80_20',
                 'purge_windows': PURGE_WINDOWS}, f)

Train : 148,202   files=13
Test  : 36,960   files=13
Purged: 260  (around train/test boundary inside each file)
Per class TRAIN: {'BENIGN': 13020, 'Amplification': 40380, 'Syn': 41230, 'UDP-lag': 6393, 'WebDDoS': 0, 'TFTP': 44441, 'Portmap': 2738}
Per class TEST : {'BENIGN': 9900, 'Amplification': 10000, 'Syn': 9170, 'UDP-lag': 973, 'WebDDoS': 0, 'TFTP': 5939, 'Portmap': 978}
Scaled.  Train mean ~0, std ~1 : -0.0010558325 1.2945745


In [23]:
# ── Cell 7: SMOTETomek for minority attack classes ─────────────────────────
# Strategy:
#   - Up-sample UDP-lag and Portmap to TARGET — they are the only true
#     minority attack classes.  Amplification/Syn/TFTP already >TARGET.
#   - BENIGN kept real — downstream models use class_weight='balanced'.
#   - WebDDoS (0 windows) cannot be synthesized: only ~439 rows in
#     CICDDoS2019, lost to majority-vote at WINDOW_SIZE=100.  Excluded as
#     a known dataset limitation (Sharafaldin 2019).
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

TARGET            = 25_000
SYNTHESIZE_BENIGN = False     # flip to True if BENIGN recall is bad after training

KEEP_REAL = set() if SYNTHESIZE_BENIGN else {0}

sampling_strategy = {}
for c in range(N_CLASSES):
    cur = int(np.sum(y_tr == c))
    if c in KEEP_REAL or cur <= 5 or cur >= TARGET:
        continue
    sampling_strategy[c] = TARGET

print('Oversample targets :', {CLASS_NAMES[c]: n for c, n in sampling_strategy.items()})

if sampling_strategy:
    min_class_size = min(int(np.sum(y_tr == c)) for c in sampling_strategy)
    k = max(1, min(5, min_class_size - 1))
    print(f'k_neighbors = {k} (smallest class has {min_class_size} samples)')
    try:
        smt = SMOTETomek(
            sampling_strategy=sampling_strategy,
            smote=SMOTE(sampling_strategy=sampling_strategy,
                        k_neighbors=k, random_state=RANDOM_SEED),
            random_state=RANDOM_SEED,
        )
        X_tr_b, y_tr_b = smt.fit_resample(X_tr_s, y_tr)
        print('Used SMOTETomek (SMOTE + Tomek-link cleaning).')
    except Exception as exc:
        print(f'[SMOTETomek failed: {exc}] — fallback to plain SMOTE.')
        smote = SMOTE(sampling_strategy=sampling_strategy,
                      k_neighbors=k, random_state=RANDOM_SEED)
        X_tr_b, y_tr_b = smote.fit_resample(X_tr_s, y_tr)
else:
    X_tr_b, y_tr_b = X_tr_s, y_tr
    print('No class below target; skipping resampling.')

print(f'\nAfter resample: {len(X_tr_b):,} rows')
print('Per class    :', {CLASS_NAMES[c]: int(np.sum(y_tr_b == c)) for c in range(N_CLASSES)})

Oversample targets : {'UDP-lag': 25000, 'Portmap': 25000}
k_neighbors = 5 (smallest class has 2738 samples)
Used SMOTETomek (SMOTE + Tomek-link cleaning).

After resample: 187,211 rows
Per class    : {'BENIGN': 12995, 'Amplification': 40375, 'Syn': 40367, 'UDP-lag': 24670, 'WebDDoS': 0, 'TFTP': 43804, 'Portmap': 25000}


---
## Section 3 — Train six baselines

In [26]:
# ── Cell 7.5: Drop empty classes and remap labels to contiguous range ──────
# XGBoost (≥2.0) requires y values to be contiguous in [0, num_class).
# WebDDoS produced 0 windows so we have a gap.  Remap once here and
# overwrite N_CLASSES / CLASS_NAMES so every downstream cell stays correct.
present_classes  = sorted(set(np.unique(y_tr_b)) | set(np.unique(y_te)))
to_contig        = {orig: new for new, orig in enumerate(present_classes)}
N_CLASSES_OLD    = N_CLASSES
CLASS_NAMES_OLD  = dict(CLASS_NAMES)

CLASS_NAMES = {to_contig[c]: CLASS_NAMES_OLD[c] for c in present_classes}
N_CLASSES   = len(present_classes)

y_tr_b = np.array([to_contig[v] for v in y_tr_b], dtype=np.int64)
y_te   = np.array([to_contig[v] for v in y_te],   dtype=np.int64)

print(f'Remap: {N_CLASSES_OLD} -> {N_CLASSES} classes')
print(f'Dropped classes: {[CLASS_NAMES_OLD[c] for c in range(N_CLASSES_OLD) if c not in present_classes]}')
print(f'New CLASS_NAMES: {CLASS_NAMES}')
print(f'Per class train:', {CLASS_NAMES[c]: int(np.sum(y_tr_b == c)) for c in range(N_CLASSES)})
print(f'Per class test :', {CLASS_NAMES[c]: int(np.sum(y_te   == c)) for c in range(N_CLASSES)})

Remap: 7 -> 6 classes
Dropped classes: ['WebDDoS']
New CLASS_NAMES: {0: 'BENIGN', 1: 'Amplification', 2: 'Syn', 3: 'UDP-lag', 4: 'TFTP', 5: 'Portmap'}
Per class train: {'BENIGN': 12995, 'Amplification': 40375, 'Syn': 40367, 'UDP-lag': 24670, 'TFTP': 43804, 'Portmap': 25000}
Per class test : {'BENIGN': 9900, 'Amplification': 10000, 'Syn': 9170, 'UDP-lag': 973, 'TFTP': 5939, 'Portmap': 978}


In [27]:
# ── Cell 8: Helper — evaluate + collect metrics ─────────────────────────────
def evaluate(name, y_true, y_pred, y_proba=None, classes=None):
    classes = classes if classes is not None else sorted(set(y_true) | set(y_pred))
    target_names = [CLASS_NAMES[c] for c in classes]
    rep = classification_report(y_true, y_pred, labels=classes,
                                target_names=target_names, digits=4, zero_division=0)
    print(f'\n=== {name} ===')
    print(rep)
    metrics = {
        'model': name,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }
    if y_proba is not None:
        try:
            # Build full-class proba aligned to N_CLASSES
            proba_full = np.zeros((len(y_true), N_CLASSES))
            for j, c in enumerate(classes):
                proba_full[:, c] = y_proba[:, j]
            metrics['roc_auc_ovr'] = float(roc_auc_score(
                y_true, proba_full, multi_class='ovr', average='macro',
                labels=list(range(N_CLASSES))
            ))
        except Exception as exc:
            metrics['roc_auc_ovr'] = float('nan')
            print(f'   (ROC-AUC skipped: {exc})')
    return metrics, rep

results = []


In [28]:
# ── Cell 9: Baseline 1 — XGBoost (GPU hist) ─────────────────────────────────
xgb_params = dict(
    objective='multi:softprob',
    num_class=N_CLASSES,
    tree_method='hist',
    device='cuda' if DEVICE.type == 'cuda' else 'cpu',
    max_depth=8,
    learning_rate=0.08,
    n_estimators=600,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
)

t0 = time.time()
xgb_clf = xgb.XGBClassifier(**xgb_params)
xgb_clf.fit(X_tr_b, y_tr_b,
            eval_set=[(X_te_s, y_te)],
            verbose=False)
print(f'XGBoost fit in {time.time()-t0:.1f}s')

xgb_pred  = xgb_clf.predict(X_te_s)
xgb_proba = xgb_clf.predict_proba(X_te_s)
m, _ = evaluate('XGBoost', y_te, xgb_pred, xgb_proba,
                classes=sorted(set(y_te) | set(xgb_pred)))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)
xgb_clf.save_model(MODELS_DIR / 'xgb.json')


XGBoost fit in 11.0s

=== XGBoost ===
               precision    recall  f1-score   support

       BENIGN     0.9972    0.9984    0.9978      9900
Amplification     0.9999    1.0000    1.0000     10000
          Syn     0.9067    0.9987    0.9505      9170
      UDP-lag     0.5455    0.0062    0.0122       973
         TFTP     0.9987    1.0000    0.9993      5939
      Portmap     0.9889    1.0000    0.9944       978

     accuracy                         0.9731     36960
    macro avg     0.9061    0.8339    0.8257     36960
 weighted avg     0.9636    0.9731    0.9608     36960



In [29]:
# ── Cell 10: Baseline 2 — LightGBM ─────────────────────────────────────────
lgb_params = dict(
    objective='multiclass',
    num_class=N_CLASSES,
    learning_rate=0.08,
    num_leaves=128,
    max_depth=-1,
    n_estimators=600,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RANDOM_SEED,
    class_weight='balanced',
    n_jobs=-1,
    device_type='gpu' if DEVICE.type == 'cuda' else 'cpu',
    verbose=-1,
)

t0 = time.time()
try:
    lgb_clf = lgb.LGBMClassifier(**lgb_params)
    lgb_clf.fit(X_tr_b, y_tr_b,
                eval_set=[(X_te_s, y_te)],
                callbacks=[lgb.early_stopping(50, verbose=False)])
except Exception as exc:
    print(f'GPU path failed ({exc}); retry CPU')
    lgb_params['device_type'] = 'cpu'
    lgb_clf = lgb.LGBMClassifier(**lgb_params)
    lgb_clf.fit(X_tr_b, y_tr_b,
                eval_set=[(X_te_s, y_te)],
                callbacks=[lgb.early_stopping(50, verbose=False)])
print(f'LightGBM fit in {time.time()-t0:.1f}s')

lgb_pred  = lgb_clf.predict(X_te_s)
lgb_proba = lgb_clf.predict_proba(X_te_s)
m, _ = evaluate('LightGBM', y_te, lgb_pred, lgb_proba,
                classes=sorted(set(y_te) | set(lgb_pred)))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)
with open(MODELS_DIR / 'lgb.pkl', 'wb') as f:
    pickle.dump(lgb_clf, f)


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LightGBM fit in 14.6s

=== LightGBM ===
               precision    recall  f1-score   support

       BENIGN     0.9970    0.9988    0.9979      9900
Amplification     1.0000    1.0000    1.0000     10000
          Syn     0.9069    0.9987    0.9506      9170
      UDP-lag     0.7778    0.0072    0.0143       973
         TFTP     0.9997    0.9995    0.9996      5939
      Portmap     0.9809    1.0000    0.9904       978

     accuracy                         0.9731     36960
    macro avg     0.9437    0.8340    0.8254     36960
 weighted avg     0.9697    0.9731    0.9609     36960



In [30]:
# ── Cell 11: Baseline 3 — RandomForest ──────────────────────────────────────
t0 = time.time()
rf_clf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2,
    class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_SEED,
)
rf_clf.fit(X_tr_b, y_tr_b)
print(f'RandomForest fit in {time.time()-t0:.1f}s')

rf_pred  = rf_clf.predict(X_te_s)
rf_proba = rf_clf.predict_proba(X_te_s)
m, _ = evaluate('RandomForest', y_te, rf_pred, rf_proba,
                classes=sorted(rf_clf.classes_))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)
with open(MODELS_DIR / 'rf.pkl', 'wb') as f:
    pickle.dump(rf_clf, f)


RandomForest fit in 45.1s

=== RandomForest ===
               precision    recall  f1-score   support

       BENIGN     0.9966    0.9992    0.9979      9900
Amplification     0.9999    1.0000    1.0000     10000
          Syn     0.9081    0.9987    0.9512      9170
      UDP-lag     0.5714    0.0082    0.0162       973
         TFTP     0.9998    1.0000    0.9999      5939
      Portmap     0.9839    1.0000    0.9919       978

     accuracy                         0.9733     36960
    macro avg     0.9100    0.8344    0.8262     36960
 weighted avg     0.9645    0.9733    0.9612     36960



In [31]:
# ── Cell 12: Baseline 4 — ExtraTrees ────────────────────────────────────────
t0 = time.time()
et_clf = ExtraTreesClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2,
    class_weight='balanced_subsample',
    n_jobs=-1, random_state=RANDOM_SEED,
)
et_clf.fit(X_tr_b, y_tr_b)
print(f'ExtraTrees fit in {time.time()-t0:.1f}s')

et_pred  = et_clf.predict(X_te_s)
et_proba = et_clf.predict_proba(X_te_s)
m, _ = evaluate('ExtraTrees', y_te, et_pred, et_proba,
                classes=sorted(et_clf.classes_))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)
with open(MODELS_DIR / 'et.pkl', 'wb') as f:
    pickle.dump(et_clf, f)


ExtraTrees fit in 8.5s

=== ExtraTrees ===
               precision    recall  f1-score   support

       BENIGN     0.9965    0.9995    0.9980      9900
Amplification     1.0000    1.0000    1.0000     10000
          Syn     0.9080    0.9987    0.9512      9170
      UDP-lag     0.7692    0.0103    0.0203       973
         TFTP     1.0000    1.0000    1.0000      5939
      Portmap     0.9859    1.0000    0.9929       978

     accuracy                         0.9735     36960
    macro avg     0.9433    0.8347    0.8271     36960
 weighted avg     0.9698    0.9735    0.9614     36960



In [32]:
# ── Cell 13: Baseline 5 — MLP ───────────────────────────────────────────────
t0 = time.time()
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu', solver='adam',
    alpha=1e-4, batch_size=512, learning_rate_init=1e-3,
    max_iter=60, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=8,
    random_state=RANDOM_SEED,
)
mlp_clf.fit(X_tr_b, y_tr_b)
print(f'MLP fit in {time.time()-t0:.1f}s')

mlp_pred  = mlp_clf.predict(X_te_s)
mlp_proba = mlp_clf.predict_proba(X_te_s)
m, _ = evaluate('MLP', y_te, mlp_pred, mlp_proba,
                classes=sorted(mlp_clf.classes_))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)
with open(MODELS_DIR / 'mlp.pkl', 'wb') as f:
    pickle.dump(mlp_clf, f)


MLP fit in 106.5s

=== MLP ===
               precision    recall  f1-score   support

       BENIGN     0.9975    0.9949    0.9962      9900
Amplification     0.9987    1.0000    0.9994     10000
          Syn     0.9047    0.9988    0.9494      9170
      UDP-lag     0.4615    0.0123    0.0240       973
         TFTP     0.9997    0.9993    0.9995      5939
      Portmap     0.9929    1.0000    0.9964       978

     accuracy                         0.9722     36960
    macro avg     0.8925    0.8342    0.8275     36960
 weighted avg     0.9609    0.9722    0.9604     36960



In [33]:
# ── Cell 14: Baseline 6 — 1D-CNN (PyTorch) ──────────────────────────────────
class Conv1DNet(nn.Module):
    def __init__(self, in_dim=N_FEATURES, n_classes=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1))  # (B, 1, F)

cnn_model = Conv1DNet().to(DEVICE)
opt = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

# class weights for imbalance — even after SMOTE, CNN benefits
counts = np.bincount(y_tr_b, minlength=N_CLASSES).astype(np.float32)
cw = counts.sum() / (counts + 1.0)
cw = cw / cw.mean()
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE))

train_ds = TensorDataset(torch.tensor(X_tr_b),  torch.tensor(y_tr_b))
test_ds  = TensorDataset(torch.tensor(X_te_s),  torch.tensor(y_te))
train_ld = DataLoader(train_ds, batch_size=1024, shuffle=True, num_workers=0)
test_ld  = DataLoader(test_ds,  batch_size=2048, shuffle=False, num_workers=0)

EPOCHS = 25
t0 = time.time()
best_f1 = -1.0
patience = 5; bad = 0
for ep in range(1, EPOCHS + 1):
    cnn_model.train()
    tot, n = 0.0, 0
    for xb, yb in train_ld:
        xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE)
        opt.zero_grad()
        logits = cnn_model(xb)
        loss = loss_fn(logits, yb)
        loss.backward(); opt.step()
        tot += loss.item() * len(xb); n += len(xb)

    cnn_model.eval()
    preds, probs = [], []
    with torch.no_grad():
        for xb, _ in test_ld:
            xb = xb.to(DEVICE)
            p = torch.softmax(cnn_model(xb), dim=1).cpu().numpy()
            probs.append(p); preds.append(p.argmax(1))
    cnn_pred  = np.concatenate(preds)
    cnn_proba = np.concatenate(probs)
    f1 = f1_score(y_te, cnn_pred, average='macro', zero_division=0)
    print(f'  epoch {ep:02d}  loss={tot/n:.4f}  macro-F1={f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1; bad = 0
        torch.save(cnn_model.state_dict(), MODELS_DIR / 'cnn.pt')
    else:
        bad += 1
        if bad >= patience:
            print(f'  early stop @ epoch {ep}')
            break

# Reload best
cnn_model.load_state_dict(torch.load(MODELS_DIR / 'cnn.pt'))
cnn_model.eval()
preds, probs = [], []
with torch.no_grad():
    for xb, _ in test_ld:
        xb = xb.to(DEVICE)
        p = torch.softmax(cnn_model(xb), dim=1).cpu().numpy()
        probs.append(p); preds.append(p.argmax(1))
cnn_pred  = np.concatenate(preds)
cnn_proba = np.concatenate(probs)

m, _ = evaluate('1D-CNN', y_te, cnn_pred, cnn_proba, classes=list(range(N_CLASSES)))
m['fit_sec'] = round(time.time() - t0, 2)
results.append(m)


  epoch 01  loss=0.9672  macro-F1=0.6655
  epoch 02  loss=0.4236  macro-F1=0.7623
  epoch 03  loss=0.3311  macro-F1=0.8031
  epoch 04  loss=0.2994  macro-F1=0.8050
  epoch 05  loss=0.2811  macro-F1=0.7989
  epoch 06  loss=0.2666  macro-F1=0.8006
  epoch 07  loss=0.2543  macro-F1=0.7976
  epoch 08  loss=0.2452  macro-F1=0.7994
  epoch 09  loss=0.2386  macro-F1=0.8052
  epoch 10  loss=0.2301  macro-F1=0.8057
  epoch 11  loss=0.2257  macro-F1=0.8402
  epoch 12  loss=0.2214  macro-F1=0.8372
  epoch 13  loss=0.2117  macro-F1=0.8335
  epoch 14  loss=0.2131  macro-F1=0.8342
  epoch 15  loss=0.2047  macro-F1=0.8390
  epoch 16  loss=0.2046  macro-F1=0.8366
  early stop @ epoch 16

=== 1D-CNN ===
               precision    recall  f1-score   support

       BENIGN     0.9981    0.9773    0.9876      9900
Amplification     0.9961    0.9877    0.9919     10000
          Syn     0.9097    0.9748    0.9411      9170
      UDP-lag     0.1569    0.0987    0.1211       973
         TFTP     0.9997    

---
## Section 4 — Comparison + best-model SHAP

In [34]:
# ── Cell 15: Results table + plot ───────────────────────────────────────────
df_res = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
print(df_res.to_string(index=False))
df_res.to_csv(RESULT_DIR / 'baseline_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_df = df_res.sort_values('macro_f1')
bar = ax.barh(plot_df['model'], plot_df['macro_f1'], color='#4C78A8')
for b, v in zip(bar, plot_df['macro_f1']):
    ax.text(v + 0.005, b.get_y() + b.get_height()/2, f'{v:.3f}', va='center')
ax.set_xlabel('Macro F1 on held-out (group-split) test set')
ax.set_xlim(0, 1.0)
ax.set_title('CICDDoS2019 — Baseline detection comparison')
plt.tight_layout()
plt.savefig(FIG_DIR / 'baseline_macro_f1.png', dpi=140)
plt.close()
print('Saved figure ->', FIG_DIR / 'baseline_macro_f1.png')


       model  accuracy  balanced_acc  macro_f1  weighted_f1  roc_auc_ovr  fit_sec
      1D-CNN  0.960444      0.839573  0.840178     0.956630     0.996264    36.27
         MLP  0.972240      0.834235  0.827487     0.960389     0.999185   106.78
  ExtraTrees  0.973485      0.834744  0.827057     0.961368     0.999726     9.03
RandomForest  0.973350      0.834351  0.826180     0.961193     0.998778    45.54
     XGBoost  0.973079      0.833874  0.825692     0.960848     0.999682    11.20
    LightGBM  0.973133      0.834028  0.825448     0.960901     0.999752    14.99
Saved figure -> /kaggle/working/pad_onap_detect/figures/baseline_macro_f1.png


In [35]:
# ── Cell 16: Per-class F1 heatmap ───────────────────────────────────────────
per_class_rows = []
for r, name in [
    (xgb_pred, 'XGBoost'),
    (lgb_pred, 'LightGBM'),
    (rf_pred,  'RandomForest'),
    (et_pred,  'ExtraTrees'),
    (mlp_pred, 'MLP'),
    (cnn_pred, '1D-CNN'),
]:
    row = {'model': name}
    for c in range(N_CLASSES):
        mask = (y_te == c)
        if mask.sum() == 0:
            row[CLASS_NAMES[c]] = float('nan')
            continue
        f = f1_score((y_te == c).astype(int), (r == c).astype(int), zero_division=0)
        row[CLASS_NAMES[c]] = f
    per_class_rows.append(row)

df_pc = pd.DataFrame(per_class_rows).set_index('model')
print(df_pc.round(3))
df_pc.to_csv(RESULT_DIR / 'per_class_f1.csv')

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(df_pc.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(df_pc.columns))); ax.set_xticklabels(df_pc.columns, rotation=30, ha='right')
ax.set_yticks(range(len(df_pc.index)));   ax.set_yticklabels(df_pc.index)
for i in range(df_pc.shape[0]):
    for j in range(df_pc.shape[1]):
        v = df_pc.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v < 0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
ax.set_title('Per-class F1 by baseline')
plt.tight_layout()
plt.savefig(FIG_DIR / 'per_class_f1_heatmap.png', dpi=140)
plt.close()


              BENIGN  Amplification    Syn  UDP-lag   TFTP  Portmap
model                                                              
XGBoost        0.998          1.000  0.950    0.012  0.999    0.994
LightGBM       0.998          1.000  0.951    0.014  1.000    0.990
RandomForest   0.998          1.000  0.951    0.016  1.000    0.992
ExtraTrees     0.998          1.000  0.951    0.020  1.000    0.993
MLP            0.996          0.999  0.949    0.024  0.999    0.996
1D-CNN         0.988          0.992  0.941    0.121  0.999    1.000


In [36]:
# ── Cell 17: SHAP on best XGBoost model ─────────────────────────────────────
# SHAP can be heavy on multi-class; subsample 2k rows from test
sub_idx = np.random.RandomState(RANDOM_SEED).choice(len(X_te_s),
                                                    size=min(2000, len(X_te_s)),
                                                    replace=False)
X_sub = X_te_s[sub_idx]

try:
    explainer = shap.TreeExplainer(xgb_clf)
    sv = explainer.shap_values(X_sub)
    # sv is list[N_CLASSES] of (n, F)  OR  ndarray (n, F, C) — handle both
    if isinstance(sv, list):
        mean_abs = np.mean([np.abs(s).mean(0) for s in sv], axis=0)
    else:
        mean_abs = np.abs(sv).mean(axis=(0, 2))
    order = np.argsort(mean_abs)[::-1]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([FEATURE_NAMES[i] for i in order[::-1]],
            [mean_abs[i] for i in order[::-1]], color='#E45756')
    ax.set_xlabel('Mean |SHAP| (averaged over classes)')
    ax.set_title('Feature importance — XGBoost (SHAP)')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'shap_xgb.png', dpi=140)
    plt.close()
    pd.DataFrame({'feature': [FEATURE_NAMES[i] for i in order],
                  'mean_abs_shap': [mean_abs[i] for i in order]}
                 ).to_csv(RESULT_DIR / 'shap_xgb.csv', index=False)
    print('SHAP done.')
except Exception as exc:
    print(f'SHAP skipped: {exc}')


SHAP done.


In [37]:
# ── Cell 18: Package outputs ────────────────────────────────────────────────
import shutil
zip_path = shutil.make_archive('/kaggle/working/pad_onap_detect', 'zip', OUTPUT_DIR)
print('Archive ->', zip_path)
print('\nFinal ranking:')
print(df_res[['model', 'macro_f1', 'balanced_acc', 'roc_auc_ovr', 'fit_sec']].to_string(index=False))


Archive -> /kaggle/working/pad_onap_detect.zip

Final ranking:
       model  macro_f1  balanced_acc  roc_auc_ovr  fit_sec
      1D-CNN  0.840178      0.839573     0.996264    36.27
         MLP  0.827487      0.834235     0.999185   106.78
  ExtraTrees  0.827057      0.834744     0.999726     9.03
RandomForest  0.826180      0.834351     0.998778    45.54
     XGBoost  0.825692      0.833874     0.999682    11.20
    LightGBM  0.825448      0.834028     0.999752    14.99
